In [1]:
import json
import pandas as pd

# 1. Les data
with open("traffic.jsonl", encoding="utf-8") as file:
    df = pd.DataFrame(json.loads(line) for line in file if line.strip())

df["departure"] = pd.to_datetime(df["depature"], format="%H:%M")
df["arrival"] = pd.to_datetime(df["arrival"], format="%H:%M")

df["duration"] = (
    (df["arrival"] - df["departure"]).dt.total_seconds() / 60
)

df["departure_minutes"] = (
    df["departure"].dt.hour * 60
    + df["departure"].dt.minute
)

# 2. Del hver rute i trening og validering
train_dfs = {}
val_dfs = {}

for route, data in df.groupby("road"):
    train = data.sample(frac=0.8, random_state=42)
    val = data.drop(index=train.index)

    train_dfs[route] = train.copy()
    val_dfs[route] = val.copy()

# 3. Finn felles min/maks fra kun treningsdataene
all_train = pd.concat(train_dfs.values())

x_min = all_train["departure_minutes"].min()
x_max = all_train["departure_minutes"].max()
x_range = x_max - x_min

print(f"Min avgangstid: {x_min} min, maks avgangstid: {x_max} min")

if x_range == 0:
    raise ValueError("Treningsdataene må ha ulike avgangstider.")

# 4. Skaler begge settene med treningens grenser
for route in train_dfs:
    for data in (train_dfs[route], val_dfs[route]):
        data["x"] = (data["departure_minutes"] - x_min) / x_range
        data["y"] = data["duration"]

# Vis antall målinger per rute
pd.DataFrame({
    "Trening": {route: len(data) for route, data in train_dfs.items()},
    "Validering": {route: len(data) for route, data in val_dfs.items()},
})

Min avgangstid: 420 min, maks avgangstid: 1019 min


,Trening,Validering
A->C->D,202,50
A->C->E,210,53
B->C->D,207,52
B->C->E,206,51


In [2]:
import plotly.express as px

for route, data in train_dfs.items():
    fig = px.scatter(
        data,
        x="x",
        y="y",
        title=f"Treningsdata: {route}",
        labels={
            "x": "Skalert avgangstid",
            "y": "Reisetid (minutter)"
        },
        hover_data=["departure_minutes"]
    )

    fig.show(renderer="vscode")

In [25]:
import numpy as np

def sample_theta_acd(size_of_theta):
    return np.random.uniform(
        low=[0, 0, 0, 60],
        high=[60, 4 * np.pi, 2 * np.pi, 140],
        size=size_of_theta
    )

In [26]:
def get_loss(y_hat, ys):
    # No change needed, returns quadratic loss.
    loss = ((y_hat - ys)**2).sum()
    return loss

In [27]:
def pred_acd(x, theta):
    a, b, c, d = theta
    return a * np.sin(b * x + c) + d

In [30]:
import tqdm 

data_acd = train_dfs["A->C->D"]
xs = data_acd["x"].to_numpy()
ys = data_acd["y"].to_numpy()
n_params = 4

best_theta = sample_theta_acd(n_params)
best_loss = float('inf')

for i in tqdm.tqdm(range(1000000)):
    if i < 10000 or np.random.random() < 0.20:
        curr_theta = sample_theta_acd(n_params)
    else:
        curr_theta = best_theta.copy()
        index = np.random.randint(n_params)

        sampled_theta = sample_theta_acd(n_params)
        curr_theta[index] = sampled_theta[index]

    y_hat = pred_acd(xs, curr_theta)
    curr_loss = get_loss(y_hat, ys)

    if curr_loss < best_loss:
        best_loss = curr_loss
        best_theta = curr_theta

print(f"Best loss: {best_loss}")
print(f"Best theta: {best_theta}")

x_plot = np.linspace(xs.min(), xs.max(), 500)

fig = px.line(
    x=x_plot,
    y=pred_acd(x_plot, best_theta),
    title="Prediksjon med beste theta",
    labels={"x": "Skalert avgangstid", "y": "Reisetid (minutter)"}
)
fig.add_scatter(x=xs, y=ys, mode="markers", name="Treningsdata")
fig.show(renderer="vscode")

rmse = np.sqrt(best_loss / len(ys))
print(f"RMSE på treningsdata: {rmse:.2f} minutter")



100%|██████████| 1000000/1000000 [00:16<00:00, 60710.71it/s]

Best loss: 5835.0805697043415
Best theta: [ 33.52533001   4.79692269   2.31996204 105.43875151]


RMSE på treningsdata: 5.37 minutter


In [31]:
val_acd = val_dfs["A->C->D"]

xs_val = val_acd["x"].to_numpy()
ys_val = val_acd["y"].to_numpy()

y_hat_val = pred_acd(xs_val, best_theta)
rmse_val = np.sqrt(np.mean((y_hat_val - ys_val) ** 2))

print(f"RMSE trening: {np.sqrt(best_loss / len(ys)):.2f} minutter")
print(f"RMSE validering: {rmse_val:.2f} minutter")

RMSE trening: 5.37 minutter
RMSE validering: 5.36 minutter
